# Chapter 6: Precognition (Thinking Step by Step)

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)


In [ ]:
# Import python's built-in regular expression library
import re
import ollama

# Retrieve the MODEL_NAME variable from the IPython store
%store -r MODEL_NAME

# prefill lets you put words in the assistant's mouth for it to continue from
def get_completion(user_prompt: str, system_prompt="", prefill=""):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    if prefill:
        messages.append({"role": "assistant", "content": prefill})
    response = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={"temperature": 0.0, "num_predict": 2000},
    )
    return response["message"]["content"]

MODEL_NAME

---

## Lesson

If someone woke you up and immediately fired several complicated questions at
you that you had to answer right away, how well would you do? Probably not as
well as if you'd been given time to **think through your answer first**.

Guess what? Large language models are the same.

**Giving models time to think step by step can make them more accurate**,
particularly for complex tasks. However, **thinking only counts when it's out
loud**. You can't ask the model to think but output only the answer — in that
case no thinking has actually happened. It needs tokens to "think".

### Examples

In the prompt below, the model has to untangle a chain of comparisons to find
the *second shortest* person. Asked to answer straight away, it tends to rush
the ordering and get it wrong.

In [ ]:
# No thinking
USER_PROMPT = "Anna is taller than Beth. Beth is taller than Cara. Cara is taller than Dina. Who is the second shortest?"
get_completion(USER_PROMPT)

To improve the model's response, let's **allow the model to think things out
first before answering**. We do this by instructing it to reason out loud
before committing — giving it room to work through the ordering step by step
instead of guessing in one shot.

In [ ]:
# Thinking
SYSTEM_PROMPT = "Before giving any direct answer you must think out loud"
USER_PROMPT = "Anna is taller than Beth. Beth is taller than Cara. Cara is taller than Dina. Who is the second shortest?"
print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

**Letting the model think first can flip an answer from wrong to right** —
often that's all a mistake needs. It won't fix everything (if the model simply
doesn't know something, thinking can't invent it), but when the model *has* the
pieces and just needs room to assemble them, this is one of your
highest-leverage techniques.

Here's another case where the direct answer tends to slip, and thinking
rescues it.

In [ ]:
# No thinking
USER_PROMPT = "Name a famous movie starring an actor who was born in the year 1954."
print(get_completion(USER_PROMPT))

Let's fix this by asking the model to think step by step, this time in `<brainstorm>` tags.

In [ ]:
# Force thinking
USER_PROMPT = "Name a famous movie starring an actor who was born in the year 1954. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."
print(get_completion(USER_PROMPT))

Always be skeptical of model output — language models can produce
confident-sounding but false facts
([hallucinations](https://en.wikipedia.org/wiki/Hallucination_(artificial_intelligence))),
especially for historical details like birth years. Whatever actor and movie
*your* model named, check the birth year yourself before trusting it. (For
example, [Denzel Washington](https://en.wikipedia.org/wiki/Denzel_Washington)
was indeed born in 1954 — but confirm the one you got rather than taking the
model's word for it.)

If you would like to experiment with the lesson prompts without changing any content above, scroll all the way to the bottom of the lesson notebook to visit the [**Example Playground**](#example-playground).

---

## Exercises
- [Exercise 6.1 - Classifying Emails](#exercise-61---classifying-emails)
- [Exercise 6.2 - Email Classification Formatting](#exercise-62---email-classification-formatting)

### Exercise 6.1 - Classifying Emails
In this exercise, we'll be instructing the model to sort emails into the following categories:										
- (A) Pre-sale question
- (B) Broken or defective item
- (C) Billing question
- (D) Other (please explain)

For the first part of the exercise, change any of the `SYSTEM_PROMPT`, `USER_PROMPT` or `PREFILL` to **make the model output the correct classification and ONLY the classification**. Your answer needs to **end with the letter (A - D) of the correct category**.

Refer to the comments beside each email in the `EMAILS` list to know which category that email should be classified under.

In [ ]:
# You are only allowed to change SYSTEM_PROMPT, USER_PROMPT and PREFILL. You may not need to change them all
# HINT: At the very least you should include the categories and {email} template
SYSTEM_PROMPT = ""
USER_PROMPT = """{email}"""
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I've been waiting 4 months for my monthly charges to end after cancelling, what's going on?", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorisations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Iterate through list of emails
for i, email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = USER_PROMPT.format(email=email)
   
    # Get the model's response
    response = get_completion(formatted_prompt, system_prompt=SYSTEM_PROMPT, prefill=PREFILL)

    # Grade the model's response
    grade = any([bool(ans == response[-1]) for ans in ANSWERS[i]])
    
    # Print the model's response
    print("--------------------------- Full prompt with variable substitutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print(f"ASSISTANT TURN\n{PREFILL}\n" if PREFILL else "", end="")
    print("------------------------------------- The model's response -------------------------------------")
    print(response)
    print("------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", "✅" if grade else "❌", "\n")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_6_1_hint; print(exercise_6_1_hint)

Still stuck? Run the cell below for an example solution.						

In [ ]:
from hints import exercise_6_1_solution; print(exercise_6_1_solution)

### Exercise 6.2 - Email Classification Formatting
In this exercise, we're going to refine the output of the above prompt to yield an answer formatted exactly how we want it. 

Use your favorite output formatting technique to make the model wrap JUST the letter of the correct classification in `<answer></answer>` tags. For instance, the answer to the first email should contain the exact string `<answer>B</answer>`.

Refer to the comments beside each email in the `EMAILS` list if you forget which letter category is correct for each email.

In [ ]:
# You are only allowed to change USER_PROMPT and PREFILL. You may not need to change both
# HINT: At the very least you should include the categories and {email} template
USER_PROMPT = """{email}"""
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I've been waiting 4 months for my monthly charges to end after cancelling, what's going on?", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "<answer>A</answer>",
    "B": "<answer>B</answer>",
    "C": "<answer>C</answer>",
    "D": "<answer>D</answer>"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = USER_PROMPT.format(email=email)

    # Get the model's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade the model's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Print the model's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print(f"ASSISTANT TURN\n{PREFILL}\n" if PREFILL else "", end="")
    print("\n------------------------------------- The model's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", "✅" if grade else "❌", "\n")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_6_2_hint; print(exercise_6_2_hint)

### Congrats!

If you've solved all exercises up until this point, you're ready to move to the next chapter. Happy prompting!

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson and tweak prompts to see how it may affect the model's responses.

In [ ]:
# No thinking
USER_PROMPT = "Anna is taller than Beth. Beth is taller than Cara. Cara is taller than Dina. Who is the second shortest?"
get_completion(USER_PROMPT)

In [ ]:
# Thinking
SYSTEM_PROMPT = "Before giving any direct answer you must think out loud"
USER_PROMPT = "Anna is taller than Beth. Beth is taller than Cara. Cara is taller than Dina. Who is the second shortest?"
print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

In [ ]:
# No thinking
USER_PROMPT = "Name a famous movie starring an actor who was born in the year 1954."
print(get_completion(USER_PROMPT))

In [ ]:
# Force thinking
USER_PROMPT = "Name a famous movie starring an actor who was born in the year 1954. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."
print(get_completion(USER_PROMPT))